In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import f1_score

In [39]:
import torch

print("Czy CUDA dostępna?:", torch.cuda.is_available())
print("Liczba dostępnych GPU:", torch.cuda.device_count())
if torch.cuda.is_available():
    print("Nazwa karty:", torch.cuda.get_device_name(0))
    # Test alokacji tensora w pamięci VRAM karty
    x = torch.ones((1000, 1000), device="cuda")
    print("Tensor pomyślnie zaalokowany w VRAM:", x.device)

Czy CUDA dostępna?: True
Liczba dostępnych GPU: 1
Nazwa karty: NVIDIA GeForce RTX 5060 Ti
Tensor pomyślnie zaalokowany w VRAM: cuda:0


## Klasa dla Dataset
Pobiera dane z dysku dopiero w momencie zapytania (lazy loading).

In [53]:
class SlakhSpectrogramDataset(Dataset):

    def __init__(self,
                 features_dir,
                 csv_name='etykiety.csv',
                 split='train',
                 label_columns = None):
        
        self.features_dir = Path(features_dir)
        csv_path = self.features_dir/csv_name

        # Wczytanie metadanych do tabeli
        df = pd.read_csv(csv_path)

        self.df = df[df["dataset"] == split].reset_index(drop = True)

        if len(self.df) == 0:
            raise ValueError(f"Brak plików w podzbiorze {split} w csv {csv_path}")

        # Nadanie nazw instrumtów
        if label_columns is None:
            self.label_cols = [col for col in self.df.columns if col not in ['dataset', 'set', 'audio_file', 'stem_name', 'file_path']]

        else:
            self.label_cols = label_columns


    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        # Ścieżka do plików
        file_name = Path(row['audio_file']).with_suffix('.npy')
        npy_path = self.features_dir / row['dataset'] / file_name

        # Wczytanie macierzy spektrogramu
        spectrogram = np.load(npy_path) # (filtry melowe, ramki czasowe)

        # Konwersja na tensor z dodaniem kanału (1, filtry melowe, ramki czasowe)
        spectrogram_tensor = torch.from_numpy(spectrogram).unsqueeze(0).float() # unsqueeze(0) powoduje dodanie wymiaru o wartości 1 na zerowej pozycji
            
        # Wektor etykiet
        labels = row[self.label_cols].values.astype(np.float32)
        label_tensor = torch.from_numpy(labels)

        return spectrogram_tensor, label_tensor



## Architektura CNN

In [54]:
class AudioCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()

        self.features = nn.Sequential(
            # Blok 1: wejście (batch_size, 1, mel_banks, frames)
            nn.Conv2d(1,32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            # Blok 2
            nn.Conv2d(32,64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            # Blok 3
            nn.Conv2d(64,128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            # Globalne uśrednianie: eliminuje zależność od zmiennej długości ramek
            nn.AdaptiveAvgPool2d((1, 1))
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)

## Trening

In [62]:
PROJECT_ROOT = (
    Path.cwd().parents[1]
) 
DATA_PATH = (
    PROJECT_ROOT
    / "spectrograms"
    / "mels80_frame1024_step512_windowhann_sr44100_4s"
)

train_data = SlakhSpectrogramDataset(
    features_dir=DATA_PATH, 
    csv_name='etykiety.csv',
    split='train'
)

val_data = SlakhSpectrogramDataset(
    features_dir=DATA_PATH,
    csv_name='etykiety.csv',
    split = 'validation'
)

train_loader = DataLoader(train_data,
                          batch_size=16,
                          shuffle=True,
                          num_workers=0, # Liczba procesów CPU w tle, dla większej trzeba przenieść dataset do pliku .py
                          pin_memory=True # Transfer danych bezpośrednio z RAM na GPU
                        )

val_loader = DataLoader(train_data,
                          batch_size=16,
                          shuffle=False,
                          num_workers=0, 
                          pin_memory=True 
                        )

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Uruchamianie treningu na {device}")

num_classes = len(train_data.label_cols)

model = AudioCNN(num_classes=num_classes).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

model.train()
total_loss = .0
for spectrograms, labels in train_loader:
    spectrograms, labels = spectrograms.to(device), labels.to(device)

    optimizer.zero_grad()
    outputs = model(spectrograms)
    loss = criterion(outputs,labels)
    
    loss.backward()
    optimizer.step()

    total_loss += loss.item()

print(f"Średni wartość Loss: {total_loss / len(train_loader):.4f}")

Uruchamianie treningu na cuda
Średni wartość Loss: 0.5505
